[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module7/09-llm-apps.ipynb)

# LLM Applications and Prompt Engineering
**Module 7 — Lesson 9 | Estimated time: 35 minutes**

> 💡 Enable GPU: Runtime → Change runtime type → GPU

## Learning Objectives
By the end of this notebook you will be able to:
- Understand the GPT model family and generation parameters
- Generate text with GPT-2 from the Hugging Face `transformers` library
- Apply zero-shot, few-shot, chain-of-thought, and role prompting
- Understand the structure of the OpenAI API (without running it)
- Use structured output / JSON mode patterns
- Build a simple chain with LangChain

In [ ]:
!pip install -q transformers langchain langchain-community

In [ ]:
import torch
import textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 1. The GPT Family — Overview

| Model | Released | Params | Context | Access |
|-------|----------|--------|---------|--------|
| GPT-2 | 2019 | 117M–1.5B | 1024 | Open |
| GPT-3 | 2020 | 175B | 4096 | API |
| GPT-3.5 (ChatGPT) | 2022 | ~175B | 4096–16k | API |
| GPT-4 | 2023 | Unknown | 8k–128k | API |
| Claude (Anthropic) | 2023 | Unknown | 200k | API |
| Gemini (Google) | 2023 | Unknown | 1M | API |
| LLaMA 2/3 (Meta) | 2023/24 | 7B–70B | 4k–128k | Open |

All modern LLMs are **autoregressive**: they predict one token at a time, sampling from the distribution over the vocabulary.

## 2. Sampling Parameters Explained

- **Temperature (T)**: scales logits before softmax. `T→0` = argmax (greedy), `T→∞` = uniform random.
- **Top-k**: keep only the k highest-probability tokens before sampling.
- **Top-p (nucleus)**: keep the smallest set of tokens whose cumulative probability ≥ p.
- **Repetition penalty**: discourages repeating the same tokens.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualise temperature effect on a toy distribution
logits = np.array([3.0, 1.5, 0.8, 0.3, -0.5])
token_labels = ['the', 'a', 'an', 'this', 'that']

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
temps = [0.2, 0.7, 1.0, 2.0]
for ax, T in zip(axes, temps):
    scaled = logits / T
    probs  = np.exp(scaled - scaled.max())
    probs /= probs.sum()
    ax.bar(token_labels, probs, color='steelblue')
    ax.set_title(f'Temperature = {T}')
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1)
plt.suptitle('Effect of Temperature on Token Probabilities', fontsize=13)
plt.tight_layout(); plt.show()

## 3. Text Generation with GPT-2

In [ ]:
# Load GPT-2 (small, 117M params — runs comfortably in Colab)
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

generator = pipeline(
    'text-generation',
    model='gpt2',
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

def generate(prompt, max_new_tokens=80, temperature=0.9, top_p=0.95, top_k=50, n=1):
    outs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        num_return_sequences=n,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    return [o['generated_text'] for o in outs]

# Basic generation
prompt = 'Python is a great programming language because'
outputs = generate(prompt, n=2)
print('=== GPT-2 Generation ===')
for i, out in enumerate(outputs):
    print(f'[{i+1}] {textwrap.fill(out, 90)}\n')

## 4. Prompt Engineering Techniques

Prompt engineering is the practice of crafting inputs to elicit better, more predictable outputs from LLMs without modifying model weights.

In [ ]:
# --- Zero-shot prompting ---
zero_shot = """Classify the sentiment of this text as POSITIVE or NEGATIVE.
Text: The food was delicious and the service was excellent.
Sentiment:"""

# --- Few-shot prompting ---
few_shot = """Classify the sentiment as POSITIVE or NEGATIVE.

Text: I loved the movie, it was fantastic!
Sentiment: POSITIVE

Text: The service was terrible and the food was cold.
Sentiment: NEGATIVE

Text: A solid performance with a few slow moments.
Sentiment:"""

# --- Chain-of-thought prompting ---
cot = """Q: A Python list has 5 elements. I add 3 more. Then I remove 2. How many remain?
Let's think step by step:
1."""

# --- Role prompting ---
role = """You are an expert Python tutor. Explain the concept of list comprehensions to a beginner in 2 sentences.
Explanation:"""

for name, prompt in [('Zero-shot', zero_shot), ('Few-shot', few_shot),
                      ('Chain-of-thought', cot), ('Role prompt', role)]:
    out = generate(prompt, max_new_tokens=50, temperature=0.7)[0]
    completion = out[len(prompt):].split('\n')[0].strip()
    print(f'[{name}]')
    print(f'  Completion: {completion}')
    print()

## 5. OpenAI API — Structure (No API Key Required)

The code below shows the structure of an OpenAI API call. Set your key as an environment variable (`OPENAI_API_KEY`) before running in production.

In [ ]:
import os

# ---- Structure demonstration (not executed without a real key) ----
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_KEY_HERE')

openai_example_code = '''
import openai

client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Chat Completions API
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system",  "content": "You are a helpful Python tutor."},
        {"role": "user",    "content": "Explain decorators in Python in 3 bullet points."},
    ],
    temperature=0.7,
    max_tokens=200,
)
print(response.choices[0].message.content)
'''

# JSON / structured output mode
json_mode_example = '''
# Structured output (JSON mode) — always returns valid JSON
response = client.chat.completions.create(
    model="gpt-4-turbo-preview",
    response_format={"type": "json_object"},
    messages=[
        {"role": "system",  "content": "Return JSON only."},
        {"role": "user",    "content": 
         "List 3 Python built-in functions as JSON: [{name, description, example}]"},
    ],
)
import json
data = json.loads(response.choices[0].message.content)
'''

print('=== OpenAI Chat API structure ===')
print(openai_example_code)
print('=== JSON / Structured output mode ===')
print(json_mode_example)

## 6. LangChain — Simple Chain Example

LangChain provides building blocks for composing LLM applications: prompts, models, parsers, and chains.

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnableLambda

# Demonstrate LangChain PromptTemplate (model-agnostic)
template = PromptTemplate(
    input_variables=['topic', 'level'],
    template=(
        'Explain {topic} to a {level} programmer in exactly 2 sentences. '
        'Use a simple analogy.'
    )
)

formatted = template.format(topic='recursion', level='beginner')
print('Formatted prompt:')
print(formatted)
print()

# Runnable chain with a mock LLM (uses GPT-2 locally)
def gpt2_llm(prompt_text):
    out = generate(prompt_text, max_new_tokens=60, temperature=0.8)[0]
    return out[len(prompt_text):].strip()

chain = template | RunnableLambda(lambda x: gpt2_llm(x.text))

result = chain.invoke({'topic': 'recursion', 'level': 'beginner'})
print('Chain output:', result[:200])

# Multi-step chain: generate → summarise
summarise_template = PromptTemplate(
    input_variables=['text'],
    template='Summarise in one sentence: {text}\nSummary:'
)
generate_template = PromptTemplate(
    input_variables=['topic'],
    template='Write a short paragraph about {topic}:\n'
)

gen_chain  = generate_template | RunnableLambda(lambda x: gpt2_llm(x.text))
sum_chain  = summarise_template | RunnableLambda(lambda x: gpt2_llm(x.text))
full_chain = gen_chain | RunnableLambda(lambda text: sum_chain.invoke({'text': text}))

final = full_chain.invoke({'topic': 'neural networks'})
print('\nMulti-step chain (generate → summarise):', final[:150])

## 7. Practical Prompting Patterns for Python Tasks

In [ ]:
# Patterns illustrated as formatted prompts (ready to use with any LLM API)

patterns = {
    'Code generation': '''
You are an expert Python developer.
Write a Python function that:
- Accepts a list of dictionaries
- Returns the dictionary with the maximum value for a given key
- Includes type hints and a docstring
Function:
''',
    'Code review': '''
Review this Python code and list up to 3 improvements:
```python
def get_data(url):
    import requests
    r = requests.get(url)
    return r.json()
```
Issues:
1.''',
    'Debugging': '''
This Python code raises a KeyError. Explain why and provide a fix.
```python
data = {"name": "Alice"}
print(data["age"])
```
Explanation:''',
    'Documentation': '''
Write a Google-style docstring for:
```python
def cosine_similarity(vec_a, vec_b):
    dot = sum(a*b for a,b in zip(vec_a, vec_b))
    norm_a = sum(a**2 for a in vec_a)**0.5
    norm_b = sum(b**2 for b in vec_b)**0.5
    return dot / (norm_a * norm_b)
```
Docstring:''',
}

for name, prompt in patterns.items():
    print(f'{'='*60}')
    print(f'Pattern: {name}')
    print(prompt.strip()[:200])
    print()

## Practice Exercises

**Exercise 1 — Temperature Sweep**
Generate completions of the same prompt with GPT-2 at temperatures 0.1, 0.5, 1.0, and 1.5. For each temperature, generate 5 completions and compute the average unique token count per completion. Plot diversity vs temperature.

**Exercise 2 — Prompt Comparison**
Design zero-shot and 5-shot prompts for a simple task (e.g. "convert Celsius to Fahrenheit" or "identify if a word is a noun or verb"). Run both prompts through GPT-2 on 10 test cases and compare accuracy.

**Exercise 3 — LangChain RAG Skeleton**
Using LangChain, build a minimal retrieval-augmented generation (RAG) skeleton: (1) create a list of 5 `Document` objects, (2) build an in-memory vector store using `FAISS` and sentence embeddings, (3) define a chain that retrieves the top-2 relevant documents for a query and includes them in the prompt context. (You do not need a paid API key — use GPT-2 as the LLM.)